# Step 4: Qwen2.5-7B + LoRA (Colab GPU)

Runs `scripts/models/run_qwen_lora_peft.py` (4-bit-quantized via
bitsandbytes, matching this adapter's QLoRA training recipe) with this
project's trained reactant/class LoRA adapter, against the 100-target JSON
from Step 1. bfloat16 weights alone (~15GB) don't fit a T4's ~14.5GB usable
memory, so `--load-in-4bit` is required there. Use a GPU runtime (Runtime >
Change runtime type > T4 GPU).

1. Upload the eval-targets JSON produced by `build_eval_targets_uspto.py` / `build_eval_targets_ord.py`.
2. Run every cell below.
3. Download `experiments.zip` and unzip it into your local repo's `experiments/` folder.


In [ ]:
import os

if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner


In [ ]:
%pip install -q -e ".[local-models,eval-runner]"
%pip install -q -U "torchao>=0.16.0"

In [ ]:
from google.colab import files

print("Upload the eval-targets JSON from Step 1 (e.g. uspto_eval_targets.json).")
uploaded = files.upload()
input_filename = next(iter(uploaded))


In [ ]:
experiment_id = ""  # @param {type:"string"}
limit = 0  # @param {type:"integer"}

experiment_id_flag = ["--experiment-id", experiment_id] if experiment_id else []
limit_flag = ["--limit", str(limit)] if limit else []


In [ ]:
!python scripts/models/run_qwen_lora_peft.py \
    --input "{input_filename}" \
    --device cuda \
    --load-in-4bit \
    {' '.join(experiment_id_flag + limit_flag)}


In [ ]:
import shutil

from google.colab import files

archive_path = shutil.make_archive("experiments", "zip", "experiments")
files.download(archive_path)
